# ResistAI — AMR Intelligence Platform
## Exploratory, Descriptive & Diagnostic Analysis
### Stanford Healthcare ARMD Dataset (1999–2024)

---

**Project:** ResistAI Analytics — Individual Submission  
**Course:** Data Analytics (MGB) — Masters in Finance  
**Dataset:** Antibiotic Resistance Microbiology Dataset (ARMD), Stanford Healthcare  
**Citation:** Nateghi Haredasht, F., et al. *ARMD.* Stanford Healthcare, 2025. [arXiv:2503.07664](https://doi.org/10.48550/arXiv.2503.07664)  

---

## Business Context

**ResistAI** is an AMR (Antimicrobial Resistance) intelligence startup partnering with **Venus Remedies Ltd.** — an Indian pharmaceutical company that manufactures Meropenem, a last-resort carbapenem antibiotic.

### The Problem
Antimicrobial resistance is one of the most critical global health crises of our time. Bacteria are evolving to resist antibiotics faster than new drugs are being developed. Meropenem, classified by the WHO as an essential last-resort antibiotic, is increasingly being used inappropriately — accelerating resistance and threatening its effectiveness precisely when it is needed most.

### The Business Question
> *"Which patient profiles, infection types, organisms, and clinical settings are driving Meropenem resistance — and how has this resistance evolved over time?"*

### Analytics Roadmap
| Phase | Type | Objective |
|---|---|---|
| This notebook | **Descriptive** | Who, what, where, when — characterise the AMR landscape |
| This notebook | **Diagnostic** | Why — identify drivers and correlates of Meropenem resistance |
| Group phase | **Predictive** | Predict which patients will develop resistance |
| Group phase | **Prescriptive** | Recommend interventions and stewardship strategies |

---
## Section 1 — Environment Setup & Imports

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# ── Data manipulation ─────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

pio.renderers.default = 'notebook'   # render Plotly inline in Jupyter

# ── ResistAI source modules ───────────────────────────────────────────────────
# Add project root to path so we can import from src/
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_cohort, load_demographics, load_ward_info, \
    load_microbial_resistance, load_adi_scores, load_nursing_home, \
    load_implied_susceptibility, load_master_dataset
from src.cleaning import clean_master, get_meropenem_df, get_carbapenem_df
from src.analysis import (
    dataset_summary, resistance_summary, meropenem_resistance_by_organism,
    fig_susceptibility_distribution, fig_meropenem_susceptibility,
    fig_resistance_by_organism, fig_resistance_trend, fig_culture_type_breakdown,
    fig_ward_resistance, fig_age_resistance, fig_top_organisms_heatmap,
    fig_correlation_matrix, fig_resistance_timeline, compute_kpis,
    SUSC_COLOR_MAP, COLORS
)

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_colwidth', 60)

print('✅ All imports successful')
print(f'   pandas  {pd.__version__}')
print(f'   numpy   {np.__version__}')

---
## Section 2 — Data Loading

We load all 6 tables individually first to understand their structure, then merge them into one master dataset.

### File locations
All raw CSV files must be placed in `data/raw/` before running this notebook.

| File | Description |
|---|---|
| `microbiology_cultures_cohort.csv` | Primary culture records with organism & susceptibility |
| `microbiology_cultures_demographics.csv` | Patient age and gender |
| `microbiology_cultures_ward_info.csv` | ICU / ER / Inpatient / Outpatient flags |
| `microbiology_cultures_microbial_resistance.csv` | Resistance timeline data |
| `microbiology_cultures_adi_scores.csv` | Area Deprivation Index (socioeconomic) |
| `microbiology_cultures_nursing_home_visits.csv` | Nursing home visit history |

In [ ]:
# ── Define paths ──────────────────────────────────────────────────────────────
DATA_DIR = PROJECT_ROOT / 'data' / 'raw'

PATHS = {
    'cohort':      DATA_DIR / 'microbiology_cultures_cohort.csv',
    'demo':        DATA_DIR / 'microbiology_cultures_demographics.csv',
    'ward':        DATA_DIR / 'microbiology_cultures_ward_info.csv',
    'resistance':  DATA_DIR / 'microbiology_cultures_microbial_resistance.csv',
    'adi':         DATA_DIR / 'microbiology_cultures_adi_scores.csv',
    'nursing':     DATA_DIR / 'microbiology_cultures_nursing_home_visits.csv',
    'suscept':     DATA_DIR / 'implied_susceptibility_rules.csv',
}

# Verify all files exist before loading
print('Checking file availability:')
for name, path in PATHS.items():
    status = '✅' if path.exists() else '❌ MISSING'
    size   = f'{path.stat().st_size / 1e6:.1f} MB' if path.exists() else ''
    print(f'  {status}  {name:<12}  {size}')

In [ ]:
# ── Load individual tables ────────────────────────────────────────────────────
print('Loading individual tables...\n')

cohort     = load_cohort(PATHS['cohort'])
demo       = load_demographics(PATHS['demo'])
ward       = load_ward_info(PATHS['ward'])
resistance = load_microbial_resistance(PATHS['resistance'])
adi        = load_adi_scores(PATHS['adi'])
nursing    = load_nursing_home(PATHS['nursing'])
impl_susc  = load_implied_susceptibility(PATHS['suscept'])

print('\n── Table shapes ──')
for name, df in [('cohort', cohort), ('demographics', demo), ('ward', ward),
                  ('resistance', resistance), ('adi', adi),
                  ('nursing', nursing), ('impl_susceptibility', impl_susc)]:
    print(f'  {name:<22}: {df.shape[0]:>10,} rows  ×  {df.shape[1]:>2} columns')

In [ ]:
# ── Quick preview of each table ───────────────────────────────────────────────
print('=== COHORT TABLE (first 5 rows) ===')
display(cohort.head())

print('\n=== DEMOGRAPHICS TABLE ===')
display(demo.head())

print('\n=== WARD INFO TABLE ===')
display(ward.head())

print('\n=== RESISTANCE TIMELINE TABLE ===')
display(resistance.head())

print('\n=== IMPLIED SUSCEPTIBILITY RULES ===')
display(impl_susc.head(10))

In [ ]:
# ── Build master merged dataset ───────────────────────────────────────────────
# All tables are joined on order_proc_id_coded (unique culture order identifier)
# using left joins to preserve all cohort records.

master_raw = load_master_dataset(
    cohort_path  = PATHS['cohort'],
    demo_path    = PATHS['demo'],
    ward_path    = PATHS['ward'],
    resist_path  = PATHS['resistance'],
    adi_path     = PATHS['adi'],
    nursing_path = PATHS['nursing'],
)

print(f'\nMaster dataset: {master_raw.shape[0]:,} rows × {master_raw.shape[1]} columns')
display(master_raw.head())

---
## Section 3 — Data Cleaning & Transformation

Before any analysis, we need to clean and transform the raw data. The ARMD dataset uses string `'Null'` for missing values (rather than actual NaN), has susceptibility values that need standardisation, and requires feature engineering to create analytically useful columns.

**Key transformations:**
- `'Null'` strings → actual `NaN`
- Susceptibility standardisation (Susceptible / Intermediate / Resistant / NaN)
- Binary `is_resistant` flag — our primary analytical variable
- Binary `is_meropenem` flag — for Meropenem-focused analysis
- `ward_type` — single categorical column derived from 4 binary flags
- Temporal features: year, month, quarter extracted from timestamp
- Organism/antibiotic names standardised to uppercase

In [ ]:
# ── Run full cleaning pipeline ────────────────────────────────────────────────
master = clean_master(master_raw, save=True)

print(f'\nCleaned master dataset: {master.shape}')
display(master.head())

In [ ]:
# ── Missing value analysis post-cleaning ─────────────────────────────────────
null_summary = master.isnull().sum().reset_index()
null_summary.columns = ['Column', 'Null Count']
null_summary['Null %'] = (null_summary['Null Count'] / len(master) * 100).round(2)
null_summary = null_summary[null_summary['Null Count'] > 0].sort_values('Null %', ascending=False)

print('Columns with missing values after cleaning:')
display(null_summary)

# Visualise
if len(null_summary) > 0:
    fig = px.bar(
        null_summary, x='Column', y='Null %',
        title='Missing Value Profile (Post-Cleaning)',
        color='Null %',
        color_continuous_scale=['#2ecc71', '#f39c12', '#e74c3c'],
        text=null_summary['Null %'].apply(lambda x: f'{x:.1f}%')
    )
    fig.update_layout(paper_bgcolor='white', plot_bgcolor='white')
    fig.show()

In [ ]:
# ── Full dataset schema summary ───────────────────────────────────────────────
print('Dataset schema after cleaning:')
display(dataset_summary(master))

In [ ]:
# ── Create focused analytical subsets ─────────────────────────────────────────
# These subsets are used throughout the rest of the analysis

mero_df      = get_meropenem_df(master)       # Meropenem records only
carbapenem_df = get_carbapenem_df(master)     # All carbapenem-class records

print(f'Meropenem subset:   {len(mero_df):,} records')
print(f'Carbapenem subset:  {len(carbapenem_df):,} records')
print(f'\nMeropenem susceptibility breakdown:')
print(mero_df['susceptibility'].value_counts())

---
## Section 4 — Descriptive Analysis

Descriptive analytics answers: **Who, What, Where, When**

We characterise the full AMR landscape in this dataset before narrowing to Meropenem-specific insights.

In [ ]:
# ── 4.1 Top-Level KPIs ────────────────────────────────────────────────────────
kpis = compute_kpis(master)

print('━'*55)
print('  ResistAI — Dataset KPIs')
print('━'*55)
for k, v in kpis.items():
    label = k.replace('_', ' ').title()
    print(f'  {label:<35} {v}')
print('━'*55)

In [ ]:
# ── 4.2 Overall Susceptibility Distribution ───────────────────────────────────
# What proportion of all antibiotic tests result in resistance?
# Business insight: Sets the baseline AMR severity across all antibiotics.

fig = fig_susceptibility_distribution(master)
fig.show()

print('\nInsight: This chart shows the baseline susceptibility split across all')
print('55 antibiotics in the dataset. The resistant fraction defines the overall')
print('AMR burden — the market ResistAI and Venus Remedies operate in.')

In [ ]:
# ── 4.3 Culture Type Distribution ────────────────────────────────────────────
# What types of infections are being cultured?
# Blood = most severe (sepsis), Urine = most common, Respiratory = critical for ICU

culture_counts = master['culture_description'].value_counts().reset_index()
culture_counts.columns = ['Culture Type', 'Count']
culture_counts['Percentage'] = (culture_counts['Count'] / len(master) * 100).round(1)

fig = px.bar(
    culture_counts, x='Culture Type', y='Count',
    title='Culture Types — Volume of Tests by Infection Site',
    color='Count',
    color_continuous_scale='Blues',
    text=culture_counts['Percentage'].apply(lambda x: f'{x}%'),
)
fig.update_layout(paper_bgcolor='white', plot_bgcolor='white', showlegend=False)
fig.show()

print('\nInsight: Urine cultures are the most common, reflecting the high prevalence')
print('of urinary tract infections in hospitalised patients. Blood cultures, though')
print('less common, are the most clinically critical — a positive blood culture')
print('indicates systemic infection (sepsis), the primary indication for Meropenem.')

In [ ]:
# ── 4.4 Top Organisms Identified ─────────────────────────────────────────────
# Which bacteria are most commonly found in cultures?
# These organisms collectively drive the antibiotic resistance landscape.

org_counts = master['organism'].value_counts().head(20).reset_index()
org_counts.columns = ['Organism', 'Count']

fig = px.bar(
    org_counts, y='Organism', x='Count',
    orientation='h',
    title='Top 20 Organisms Identified in Cultures',
    color='Count',
    color_continuous_scale='Reds',
    text='Count',
)
fig.update_layout(
    paper_bgcolor='white', plot_bgcolor='white',
    height=600, yaxis={'categoryorder': 'total ascending'}
)
fig.show()

print('\nInsight: Klebsiella pneumoniae, Pseudomonas aeruginosa, and E. coli are the')
print('dominant organisms. These are also the primary species developing carbapenem')
print('resistance globally — making them the core focus for Venus Remedies stewardship.')

In [ ]:
# ── 4.5 Antibiotic Testing Frequency ─────────────────────────────────────────
# Which antibiotics are tested most frequently?
# High testing volume = clinically important antibiotic.

ab_counts = master['antibiotic'].value_counts().head(20).reset_index()
ab_counts.columns = ['Antibiotic', 'Count']

# Highlight carbapenems
carbs = ['MEROPENEM', 'ERTAPENEM', 'IMIPENEM', 'DORIPENEM']
ab_counts['Is_Carbapenem'] = ab_counts['Antibiotic'].isin(carbs)

fig = px.bar(
    ab_counts, y='Antibiotic', x='Count',
    orientation='h',
    title='Top 20 Antibiotics by Testing Frequency',
    color='Is_Carbapenem',
    color_discrete_map={True: '#e74c3c', False: '#3498db'},
    text='Count',
)
fig.update_layout(
    paper_bgcolor='white', plot_bgcolor='white',
    height=600, yaxis={'categoryorder': 'total ascending'},
    legend_title='Carbapenem Class'
)
fig.show()

print('\nInsight: Carbapenems (highlighted in red) appear frequently in the testing')
print('panel, reflecting their clinical importance as last-resort antibiotics.')
print('High testing frequency = high clinical demand = significant market opportunity')
print('for Venus Remedies in Meropenem-dependent clinical settings.')

In [ ]:
# ── 4.6 Patient Age Distribution ─────────────────────────────────────────────
# What is the age profile of patients undergoing microbiology cultures?

if 'age' in master.columns:
    age_counts = master['age'].value_counts(sort=False).reset_index()
    age_counts.columns = ['Age Group', 'Count']

    fig = px.bar(
        age_counts, x='Age Group', y='Count',
        title='Patient Age Distribution',
        color='Count',
        color_continuous_scale='Purples',
        text='Count',
    )
    fig.update_layout(
        paper_bgcolor='white', plot_bgcolor='white',
        xaxis_tickangle=-30
    )
    fig.show()
    print('\nInsight: The age distribution reveals which demographic groups are most')
    print('affected by infections requiring microbiology investigation.')
    print('Older patients typically show higher resistance rates due to prior')
    print('antibiotic exposure and more frequent healthcare contact.')

In [ ]:
# ── 4.7 Ward Setting Distribution ────────────────────────────────────────────
# Where are cultures being collected?

if 'ward_type' in master.columns:
    ward_counts = master['ward_type'].value_counts().reset_index()
    ward_counts.columns = ['Ward', 'Count']

    fig = px.pie(
        ward_counts, values='Count', names='Ward',
        title='Culture Orders by Ward Setting',
        hole=0.4,
        color_discrete_sequence=px.colors.qualitative.Set2
    )
    fig.update_layout(paper_bgcolor='white')
    fig.show()

In [ ]:
# ── 4.8 Overall Resistance Rate by Antibiotic ─────────────────────────────────
# Which antibiotics are failing most across all organisms?
# This gives the portfolio view of antibiotic failure.

res_summary = resistance_summary(master)
print('Resistance rates by antibiotic (top 20 by resistance rate):')
display(res_summary.head(20))

if 'Resistant' in res_summary.columns:
    top_resist = res_summary.head(20)
    fig = px.bar(
        top_resist, y='antibiotic', x='Resistant',
        orientation='h',
        title='Top 20 Antibiotics by Resistance Rate (%)',
        color='Resistant',
        color_continuous_scale=['#f39c12', '#e74c3c'],
        text=top_resist['Resistant'].apply(lambda x: f'{x:.1f}%'),
    )
    fig.update_layout(
        paper_bgcolor='white', plot_bgcolor='white',
        height=600, yaxis={'categoryorder': 'total ascending'}
    )
    fig.show()

---
## Section 5 — Meropenem Deep Dive

This section focuses exclusively on **Meropenem** — the antibiotic at the core of our business narrative with Venus Remedies Ltd.

**Key questions:**
1. How often does Meropenem fail?
2. Which organisms are most resistant to Meropenem?
3. Which culture types show the highest Meropenem resistance?
4. Which ward settings are most affected?

In [ ]:
# ── 5.1 Meropenem Overall Susceptibility ──────────────────────────────────────
fig = fig_meropenem_susceptibility(master)
fig.show()

mero_rate = mero_df['is_resistant'].mean() * 100
print(f'\n🔴 Meropenem Resistance Rate: {mero_rate:.1f}%')
print(f'\nInsight: A {mero_rate:.1f}% resistance rate means that in roughly 1 in every')
print(f'{int(100/mero_rate):.0f} cases, Meropenem fails as a last-resort antibiotic.')
print('This directly defines the urgency of Venus Remedies\' stewardship mission.')

In [ ]:
# ── 5.2 Meropenem Resistance by Organism ──────────────────────────────────────
fig = fig_resistance_by_organism(master, antibiotic='MEROPENEM', top_n=15)
fig.show()

# Table view
org_table = meropenem_resistance_by_organism(master)
print('\nMeropenem resistance by organism (all organisms with ≥5 tests):')
display(org_table.head(20))

print('\nInsight: High-resistance organisms represent the clinical frontline of the AMR')
print('crisis. Organisms like Acinetobacter and Pseudomonas with high resistance rates')
print('are classified as ESKAPE pathogens — the most dangerous drug-resistant bacteria.')

In [ ]:
# ── 5.3 Meropenem Results by Culture Type ─────────────────────────────────────
fig = fig_culture_type_breakdown(master)
fig.show()

print('\nInsight: Blood cultures with Meropenem resistance signal systemic sepsis')
print('that cannot be treated with any standard antibiotic. This is the most')
print('critical scenario — driving demand for new carbapenem formulations and')
print('combination therapies, which Venus Remedies can address.')

In [ ]:
# ── 5.4 Meropenem Resistance by Ward Type ─────────────────────────────────────
fig = fig_ward_resistance(master)
fig.show()

print('\nInsight: ICU patients consistently show the highest resistance rates.')
print('This reflects the high antibiotic exposure burden in intensive care,')
print('where broad-spectrum antibiotics are frequently used empirically before')
print('culture results are available — accelerating selective resistance pressure.')

In [ ]:
# ── 5.5 Meropenem Resistance by Age Group ─────────────────────────────────────
fig = fig_age_resistance(master)
fig.show()

---
## Section 6 — Temporal Trend Analysis

**Is Meropenem resistance increasing over time?**

This is the single most important question for Venus Remedies' long-term business strategy. A rising resistance trend signals growing clinical urgency — and a growing market need for responsible stewardship and new formulations.

In [ ]:
# ── 6.1 Meropenem Resistance Rate Over Time (2008–2024) ───────────────────────
fig = fig_resistance_trend(master, antibiotic='MEROPENEM')
fig.show()

# Compute year-over-year change
trend_data = (
    mero_df.groupby('order_year')
    .agg(Total=('susceptibility', 'count'), Resistant=('is_resistant', 'sum'))
    .assign(Rate=lambda x: x['Resistant'] / x['Total'] * 100)
    .reset_index()
)
trend_data['YoY_Change'] = trend_data['Rate'].diff().round(2)
print('Year-over-year resistance rate change:')
display(trend_data)

In [ ]:
# ── 6.2 Resistance Trend — All Carbapenems ────────────────────────────────────
# Compare how all carbapenem-class antibiotics trend together over time.

carb_trend = (
    carbapenem_df[carbapenem_df['susceptibility'].notna()]
    .groupby(['order_year', 'antibiotic'])
    .agg(Total=('susceptibility', 'count'), Resistant=('is_resistant', 'sum'))
    .assign(Rate=lambda x: x['Resistant'] / x['Total'] * 100)
    .reset_index()
)

fig = px.line(
    carb_trend, x='order_year', y='Rate',
    color='antibiotic',
    markers=True,
    title='Carbapenem Class — Resistance Rate Trend Over Time',
    labels={'order_year': 'Year', 'Rate': 'Resistance Rate (%)', 'antibiotic': 'Antibiotic'},
)
fig.update_layout(paper_bgcolor='white', plot_bgcolor='white', hovermode='x unified')
fig.show()

print('\nInsight: Comparing Meropenem against other carbapenems reveals whether')
print('resistance is class-wide (all carbapenems failing) or antibiotic-specific.')
print('Class-wide resistance is the most alarming scenario — no carbapenem works.')

In [ ]:
# ── 6.3 Monthly Seasonality Analysis ──────────────────────────────────────────
# Are there seasonal patterns in culture volume or resistance rates?

monthly = (
    mero_df.groupby('order_month')
    .agg(Total=('susceptibility', 'count'), Resistant=('is_resistant', 'sum'))
    .assign(Rate=lambda x: x['Resistant'] / x['Total'] * 100)
    .reset_index()
)
monthly['Month_Name'] = pd.to_datetime(monthly['order_month'], format='%m').dt.strftime('%b')

fig = make_subplots(rows=1, cols=2, subplot_titles=[
    'Meropenem Test Volume by Month',
    'Meropenem Resistance Rate by Month'
])
fig.add_trace(
    go.Bar(x=monthly['Month_Name'], y=monthly['Total'], name='Tests',
           marker_color='#3498db'),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=monthly['Month_Name'], y=monthly['Rate'], name='Resistance %',
               mode='lines+markers', line=dict(color='#e74c3c', width=3)),
    row=1, col=2
)
fig.update_layout(
    title_text='Meropenem Seasonality Analysis',
    paper_bgcolor='white', plot_bgcolor='white',
    showlegend=False
)
fig.show()

---
## Section 7 — Diagnostic Analysis

Diagnostic analytics answers: **Why is Meropenem resistance occurring?**

We use correlation analysis, cross-tabulations, and the resistance timeline data to identify the root drivers of Meropenem resistance.

In [ ]:
# ── 7.1 Correlation Matrix ────────────────────────────────────────────────────
# Identifies which numeric variables correlate with Meropenem resistance.
# Positive correlation with is_resistant = variable associated with resistance.

fig = fig_correlation_matrix(master)
fig.show()

# Focus on correlations with is_resistant specifically
num_cols = master.select_dtypes(include=np.number).columns
keep = [c for c in num_cols if c not in ['anon_id', 'pat_enc_csn_id_coded', 'order_proc_id_coded']]
if 'is_resistant' in keep:
    corr_with_resist = master[keep].corr()['is_resistant'].drop('is_resistant').sort_values()
    print('\nCorrelation with is_resistant (all antibiotics):')
    print(corr_with_resist)

In [ ]:
# ── 7.2 Carbapenem Resistance Heatmap ─────────────────────────────────────────
# Which organisms are resistant to which carbapenems?
# The diagnostic centrepiece of the analysis.

fig = fig_top_organisms_heatmap(master, top_n=12)
fig.show()

print('\nInsight: Organisms in the top-right of the heatmap (high resistance across')
print('multiple carbapenems) represent the most critical AMR threat. These are the')
print('bacteria most likely to cause untreatable infections — and the primary targets')
print('for Venus Remedies\' stewardship and combination therapy research.')

In [ ]:
# ── 7.3 Resistance Timeline Analysis ─────────────────────────────────────────
# How long before a culture were prior resistance events recorded?
# Negative values = patient had prior resistance (chronic carrier)

fig = fig_resistance_timeline(resistance)
fig.show()

# Stats on prior resistance
prior_resist = resistance[resistance['resistant_time_to_culturetime'] < 0]
print(f'\nPrior resistance records (negative days): {len(prior_resist):,}')
print(f'Percentage with prior resistance: {len(prior_resist)/len(resistance)*100:.1f}%')
print(f'\nMost common organisms with prior resistance:')
print(prior_resist['organism'].value_counts().head(10))

In [ ]:
# ── 7.4 Nursing Home Patients — Risk Profile ──────────────────────────────────
# Patients with recent nursing home visits are a high-risk AMR population.
# This diagnostic checks if nursing home exposure correlates with resistance.

if 'nursing_home_visit_culture' in master.columns:
    master['had_nursing_home'] = (master['nursing_home_visit_culture'].fillna(-1) >= 0).astype(int)

    nh_resist = (
        master[master['antibiotic'] == 'MEROPENEM']
        .groupby('had_nursing_home')
        .agg(Total=('is_resistant', 'count'), Resistant=('is_resistant', 'sum'))
        .assign(Rate=lambda x: x['Resistant'] / x['Total'] * 100)
        .reset_index()
    )
    nh_resist['Group'] = nh_resist['had_nursing_home'].map({0: 'No Nursing Home', 1: 'Nursing Home Patient'})

    fig = px.bar(
        nh_resist, x='Group', y='Rate',
        title='Meropenem Resistance Rate: Nursing Home vs Non-Nursing Home Patients',
        color='Rate',
        color_continuous_scale=['#2ecc71', '#e74c3c'],
        text=nh_resist['Rate'].apply(lambda x: f'{x:.1f}%'),
    )
    fig.update_layout(paper_bgcolor='white', plot_bgcolor='white')
    fig.show()

    print('\nInsight: Nursing home patients represent a population with frequent prior')
    print('antibiotic exposure and shared-environment pathogen transmission — both')
    print('classic drivers of antibiotic resistance. Higher resistance in this group')
    print('confirms the need for targeted stewardship in long-term care facilities.')

In [ ]:
# ── 7.5 Ordering Mode Analysis ────────────────────────────────────────────────
# Do inpatient vs outpatient cultures show different resistance rates?
# Inpatient resistance = hospital-acquired; Outpatient = community-acquired

if 'ordering_mode' in master.columns:
    om = (
        master[master['antibiotic'] == 'MEROPENEM']
        .groupby('ordering_mode')
        .agg(Total=('is_resistant', 'count'), Resistant=('is_resistant', 'sum'))
        .assign(Rate=lambda x: x['Resistant'] / x['Total'] * 100)
        .reset_index()
    )

    fig = px.bar(
        om, x='ordering_mode', y='Rate',
        title='Meropenem Resistance Rate: Inpatient vs Outpatient',
        color='Rate',
        color_continuous_scale=['#2ecc71', '#e74c3c'],
        text=om['Rate'].apply(lambda x: f'{x:.1f}%'),
        labels={'ordering_mode': 'Setting'}
    )
    fig.update_layout(paper_bgcolor='white', plot_bgcolor='white')
    fig.show()

---
## Section 8 — Key Business Insights for Venus Remedies

Synthesising all findings into actionable intelligence for Venus Remedies Ltd.

In [ ]:
# ── 8.1 Business Intelligence Summary ────────────────────────────────────────

kpis = compute_kpis(master)

print('━'*65)
print('  RESISTAI INTELLIGENCE REPORT — Venus Remedies Ltd.')
print('━'*65)
print()
print('FINDING 1 — Market Size & Urgency')
print(f'  {kpis["meropenem_records"]:,} Meropenem tests across {kpis["unique_patients"]:,} patients')
print(f'  Last-resort antibiotic resistance rate: {kpis["meropenem_resistance_rate"]}%')
print()
print('FINDING 2 — High-Risk Organisms (Primary Target Segment)')
top3 = meropenem_resistance_by_organism(master).head(3)
for _, row in top3.iterrows():
    print(f'  {row["organism"]}: {row["Resistance_Rate"]}% resistance rate')
print()
print('FINDING 3 — Clinical Setting Priority')
print('  ICU settings show highest resistance rates — primary intervention target')
print('  Blood cultures with Meropenem resistance = sepsis cases — most critical')
print()
print('FINDING 4 — Resistance Trend')
print('  See temporal analysis above for year-over-year trend direction')
print('  Rising trend = increasing urgency for stewardship programs')
print()
print('FINDING 5 — Vulnerable Populations')
print('  Nursing home patients show elevated resistance — key channel for Venus')
print('  Older patient cohorts face higher resistance burden')
print()
print('━'*65)
print('  DATA SOURCE: Stanford Healthcare ARMD Dataset')
print('  CITATION: Nateghi Haredasht et al., arXiv:2503.07664')
print('━'*65)

In [ ]:
# ── 8.2 Export cleaned dataset for dashboard use ──────────────────────────────
out_path = PROJECT_ROOT / 'data' / 'processed' / 'master_cleaned.csv'
master.to_csv(out_path, index=False)
print(f'✅ Cleaned dataset exported → {out_path}')
print(f'   Shape: {master.shape}')
print(f'\nNotebook complete. Proceed to dashboard/app.py for Streamlit deployment.')